<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

Install the required libraries for fetching market data, accessing academic factor datasets, and running regression analysis.

In [ ]:
!pip install numpy pandas pandas_datareader yfinance statsmodels

## Imports and setup

numpy handles array math and covariance calculations. pandas provides DataFrames for organizing return series. pandas_datareader pulls Fama-French factor data directly from Kenneth French's data library. yfinance downloads historical stock prices from Yahoo Finance. statsmodels supplies OLS regression and rolling regression tools for measuring factor sensitivities over time.

In [ ]:
import numpy as np
import pandas as pd
import pandas_datareader as pdr
import yfinance as yf
import statsmodels.api as sm
from statsmodels import regression
from statsmodels.regression.rolling import RollingOLS

## Load factor data and stock prices

Fetch the Fama-French three-factor dataset starting from 2000, which contains monthly returns for the market, size (SMB), and value (HML) factors published by Kenneth French's data library.

In [ ]:
factors = pdr.get_data_famafrench(
    'F-F_Research_Data_Factors',
    start='2000-01-01'
)[0][1:]

SMB = factors.SMB
HML = factors.HML

SMB (Small Minus Big) measures the return difference between small-cap and large-cap stocks each month. HML (High Minus Low) measures the return difference between value stocks and growth stocks. These two series are the market forces we want to test our portfolio against.

Download monthly closing prices for SPY (our benchmark) and three tech stocks, then compute percentage returns and convert to monthly periods so they align with the Fama-French data.

In [ ]:
data = yf.download(
    ['SPY', 'MSFT', 'AAPL', 'INTC'],
    start="2000-01-01",
    interval="1mo"
)['Close']

monthly_returns = data.pct_change().to_period("M")

Aligning time periods between your portfolio returns and the factor data is essential. If the frequencies don't match, the regression will either fail or produce misleading results. Using monthly periods here ensures a clean join later.

## Compute active returns and run regression

Separate SPY as the benchmark, average the remaining stock returns into an equal-weight portfolio, and subtract the benchmark to get active returns, which represent the portion of performance not explained by simply holding the market.

In [ ]:
bench = monthly_returns.pop("SPY")
R = monthly_returns.mean(axis=1)
active = R - bench

Active returns isolate what your specific stock picks contributed beyond broad market movement. This is the signal we actually want to decompose. If you skip this step and regress raw returns, you'd be measuring total market exposure rather than the unique bets your portfolio is making.

Combine active returns with the SMB and HML factor series into a single DataFrame, dropping any months where data is missing, then run OLS regression to estimate how sensitive our active returns are to each factor.

In [ ]:
df = pd.DataFrame({
    'R': active,
    'F1': SMB,
    'F2': HML,
}).dropna()

b1, b2 = regression.linear_model.OLS(
    df.R,
    df[['F1', 'F2']]
).fit().params

print(
    f'Sensitivities of active returns to factors:\n'
    f'SMB: {b1}\nHML: {b2}'
)

The coefficients b1 and b2 are your portfolio's factor sensitivities, often called factor loadings or betas. A positive SMB loading means your portfolio tilts toward smaller companies relative to SPY. A negative HML loading means it tilts toward growth stocks rather than value stocks. These numbers turn a vague sense of "I own tech stocks" into a precise measurement of what forces are actually moving your returns.

Run a 12-month rolling regression to see how your portfolio's sensitivity to each factor changes over time, then plot the rolling coefficients.

In [ ]:
exog_vars = ["SMB", "HML"]
exog = sm.add_constant(factors[exog_vars])
rols = RollingOLS(df.R, exog, window=12)
rres = rols.fit()
fig = rres.plot_recursive_coefficient(variables=exog_vars)

A single regression gives you one average sensitivity across the entire period, but markets shift. Rolling regression reveals whether your exposure to size or value has been growing, shrinking, or flipping direction. Professionals use this to catch drift in their portfolios before it becomes a problem.

## Decompose portfolio risk by factor

Use the factor covariance matrix and the regression coefficients to calculate each factor's marginal contribution to the total variance of active returns, expressed as a fraction of total active risk.

In [ ]:
F1 = df.F1
F2 = df.F2

cov = np.cov(F1, F2)
ar_squared = (active.std()) ** 2
mcar1 = (
    b1 * (b2 * cov[0, 1] + b1 * cov[0, 0])
) / ar_squared
mcar2 = (
    b2 * (b1 * cov[0, 1] + b2 * cov[1, 1])
) / ar_squared
print(f'SMB risk contribution: {mcar1}')
print(f'HML risk contribution: {mcar2}')
print(f'Unexplained risk contribution: {1 - (mcar1 + mcar2)}')

This is the payoff of the entire analysis. Instead of just knowing your factor sensitivities, you now know what percentage of your portfolio's active risk each factor actually accounts for. The unexplained portion represents risk from stock-specific events, sector dynamics, or other forces not captured by the two-factor model. If the unexplained share is large, it means your portfolio's behavior is driven by something beyond size and value, which could be a good thing if intentional or a warning sign if not. This decomposition is exactly how professional risk managers decide whether a portfolio's risk profile matches the bets they intended to make.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.